16_diffbot_EDA.ipynb
--------------
Análisis exploratorio de los datos extraídos de Google News y de Diffbot.
    
Make sure to set your Diffbot API key in a .env file as follows:
    DIFFBOT_API_KEY=your_diffbot_api_key

In [1]:
import requests
import json

from dotenv import load_dotenv
import os
load_dotenv()  # Automatically finds .env file
api_key = os.getenv('DIFFBOT_API_KEY')


## Pre-processing

In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
base_dir = Path.cwd()
csv_path = base_dir / ".." / "data" / "Monitoreo noticias with articles.csv"

df = pd.read_csv(csv_path)
print(f"Number of rows in DataFrame: {len(df)}")

Number of rows in DataFrame: 2340


In [ ]:
df = df[:700]

In [ ]:
# Cell A
def safe_json_loads(value):
    if pd.isna(value):
        return {}
    text = str(value).strip()
    if not text:
        return {}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}


def extract_diffbot_response_fields(raw_json):
    data = safe_json_loads(raw_json)
    if not isinstance(data, dict):
        return pd.Series(dtype=object)

    obj = (data.get("objects") or [{}])[0]
    req = data.get("request") or {}

    if not isinstance(obj, dict):
        obj = {}
    if not isinstance(req, dict):
        req = {}

    tags = obj.get("tags") or []
    categories = obj.get("categories") or []

    tag_labels = [tag.get("label") for tag in tags if isinstance(tag, dict) and tag.get("label")]
    tag_scores = [tag.get("score") for tag in tags if isinstance(tag, dict) and tag.get("score") is not None]
    category_names = [category.get("name") for category in categories if isinstance(category, dict) and category.get("name")]
    category_scores = [category.get("score") for category in categories if isinstance(category, dict) and category.get("score") is not None]
    category_ids = [category.get("id") for category in categories if isinstance(category, dict) and category.get("id")]

    flattened = {}
    for key, value in obj.items():
        if key in {"tags", "categories"}:
            continue
        flattened[f"diffbot_{key}"] = value
    flattened["diffbot_tag_labels"] = tag_labels
    flattened["diffbot_tag_scores"] = tag_scores
    flattened["diffbot_category_names"] = category_names
    flattened["diffbot_category_scores"] = category_scores
    flattened["diffbot_category_ids"] = category_ids
    flattened.update({f"diffbot_request_{key}": value for key, value in req.items()})

    return pd.Series(flattened)

# Apply to your dataframe
extracted = df["diffbot_response"].apply(extract_diffbot_response_fields)
df2 = pd.concat([df, extracted], axis=1)

# Preview
df2.filter(regex=r"^diffbot(_request)?_|^diffbot_(tag_labels|tag_scores|category_names|category_scores|category_ids)$").head(5)


In [ ]:
df2[["diffbot_tag_labels", "diffbot_category_names"]].head(5)

In [ ]:
df2.columns

## Exploratory Data Analysis


Missing values

In [ ]:
missing_counts_df2 = (
    df2.isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_values")
    .sort_values("missing_values", ascending=False)
)

missing_counts_df2


Validado?

In [ ]:
df2['Validado'].unique()

Sitios

In [ ]:
siteName_counts = (
    df2["diffbot_siteName"]
    .value_counts(dropna=False)
    .rename_axis("siteName")
    .reset_index(name="occurrences")
)

In [ ]:
min_occurrences = 5

chart_df = siteName_counts[
    siteName_counts["siteName"].notna() & (siteName_counts["occurrences"] >= min_occurrences)
].sort_values("occurrences")

import matplotlib.pyplot as plt

plt.figure(figsize=(12, max(6, 0.35 * len(chart_df))))
plt.barh(chart_df["siteName"], chart_df["occurrences"], color="#2c7fb8")
plt.xlabel("Occurrences")
plt.ylabel("siteName")
plt.title(f"siteName frequency for occurrences >= {min_occurrences}")
plt.tight_layout()
plt.show()


¿Cuáles no tienen sitio registrado?

Error: Timeout

In [ ]:
# Missing siteName values
missing_siteName_df = df2.loc[
    df2["diffbot_siteName"].isna(),
    ["ID", "Título", "Validado", "diffbot_response"],
].copy()

missing_siteName_df


In [ ]:
print(json.loads(missing_siteName_df.iloc[0]["diffbot_response"])['error'])

Types of Diffbot responses.

In [ ]:
df2['diffbot_response'].sample(20)

In [ ]:
diffbot_response_prefix_unique = (
    df2["diffbot_response"]
    .astype(str)
    .str[:23]
    .drop_duplicates()
    .tolist()
)

diffbot_response_prefix_unique


### Sites with anti-bot protection.

In [ ]:
rows_with_bot_in_html_df = df2.loc[
    df2["diffbot_html"].fillna("").str.contains(r"bot(?:\s|\.)", case=False, regex=True),
    ["ID", "diffbot_siteName", "diffbot_html"],
].copy()

rows_with_bot_in_html_df


### Save to CSV

In [ ]:
output_csv_path = base_dir / ".." / "data" / "Monitoreo noticias with diffbot fields.csv"
df2.to_csv(output_csv_path, index=False)
